In [98]:
import os
import io
import ast
import fitz
import json
import torch
import base64
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from FlagEmbedding import LayerWiseFlagLLMReranker


from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [92]:
load_dotenv()

True

In [100]:
JINA_API_KEY = os.getenv("JINA_API_KEY")

In [328]:
with open("idf_map.json", "r") as f:
    idf_map = json.load(f)

In [329]:
idf_map

{'a': 0.578077850775158,
 'guide': 0.7691330875378674,
 'gov': 0.4554755286828257,
 'into': 1.9218125974762528,
 'vendors': 0.24783616390458127,
 'image': 0.13005312824819776,
 'login': 0.5355182363563621,
 'log': 1.3156767939059373,
 'portal': 3.0204248861443626,
 'to': 0.2795848622191615,
 'user': 1.074514737089049,
 'this': 0.7691330875378674,
 'e': 0.7178397931503168,
 'individual': 1.148622709242771,
 'without': 1.5163474893680884,
 'singpass': 0.8232003088081431,
 'applicable': 0.7178397931503168,
 'or': 1.074514737089049,
 'sole': 1.7676619176489945,
 'pas': 2.327277705584417,
 'as': 0.8232003088081431,
 'type': 2.327277705584417,
 'singpas': 2.327277705584417,
 'societies': 1.7676619176489945,
 'password': 1.410986973710262,
 'method': 0.9409833444645266,
 'personal': 1.410986973710262,
 'for': 0.050010420574661416,
 '1': 0.9409833444645266,
 'payment': 1.410986973710262,
 'word': 2.327277705584417,
 'entity': 0.8803587226480917,
 'but': 1.634130525024472,
 'freelancers': 1.410

In [161]:
pdf_path = "pdfs\smartnation2-report.pdf"

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_20280\1914289470.py:1: SyntaxWarning: invalid escape sequence '\s'
  pdf_path = "pdfs\smartnation2-report.pdf"


In [3]:
load_dotenv()

True

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [95]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [7]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


Loading weights: 100%|██████████| 392/392 [00:01<00:00, 235.52it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [9]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [10]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().numpy()
    return coarse_vector, embeddings


In [12]:
def get_query_weights(query_tokens):
    weights = []
    for t in query_tokens:
        clean_t = t.replace('Ġ', '').lower().strip()
        w = idf_map.get(clean_t, 1.0)
        if any(char.isdigit() for char in clean_t) or '$' in clean_t:
            w *= 1.5
        weights.append(w)
    return np.array(weights, dtype=np.float32)

In [55]:
def coarse_search(coarse_vector, top_k = 50):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT id, page_number FROM pages
                ORDER BY coarse_vector <=> %s::vector
                LIMIT %s

            ''', (coarse_vector, top_k))
            dic = {}
            for row in cur.fetchall():
                dic[row[0]] = row[1]

            return dic
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [ ]:
def precision_search(query_embeddings, ids, top_k=20):
    try:
        all_page_data = {} 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        results = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32)
            
            # 1. Similarity Matrix: (Query Tokens x Page Patches)
            sim_matrix = query_embeddings @ page_matrix.T 
            
            # 2. MaxSim: Find best patch for every query token
            token_scores = np.max(sim_matrix, axis=1) 
            
            # 3. Sum: Final scalar score is the unweighted sum of MaxSim scores
            final_score = np.sum(token_scores)
            
            results.append((p_id, float(final_score)))

        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []

In [43]:
def hybrid_precision_search(query_text, query_embeddings, candidate_ids, top_k = 3):
    """
    Combines ColQwen MaxSim and Postgres FTS using RRF.
    """
    k_rrf = 60  # Standard RRF constant to balance top vs bottom ranks
    rrf_scores = {}

    try:
        with conn.cursor() as cur:
            # --- STEP 1: BM25 / KEYWORD SEARCH ---
            # We rank the candidate pages based on exact keyword matches
            cur.execute('''
                SELECT id, ts_rank_cd(fts_tokens, plainto_tsquery('english', %s)) AS rank
                FROM pages
                WHERE id IN %s
                ORDER BY rank DESC
            ''', (query_text, tuple(candidate_ids)))
            
            bm25_results = cur.fetchall()
            for rank, (p_id, score) in enumerate(bm25_results):
                if score > 0: # Only count if there's an actual keyword match
                    rrf_scores[p_id] = rrf_scores.get(p_id, 0) + (1.0 / (k_rrf + rank + 1))

            # --- STEP 2: COLQWEN MAXSIM SEARCH ---
            # Get the patch embeddings for these candidates
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(candidate_ids),))
            
            # Group patches by page
            all_page_data = {}
            for p_id, emb in cur.fetchall():
                if p_id not in all_page_data: all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

            colqwen_list = []
            for p_id, patches in all_page_data.items():
                page_matrix = np.array(patches, dtype=np.float32)
                # Pure MaxSim calculation
                sim_matrix = query_embeddings @ page_matrix.T 
                final_score = np.sum(np.max(sim_matrix, axis=1))
                colqwen_list.append((p_id, final_score))

            # Sort ColQwen results by score to get their rank
            colqwen_list.sort(key=lambda x: x[1], reverse=True)
            for rank, (p_id, _) in enumerate(colqwen_list):
                rrf_scores[p_id] = rrf_scores.get(p_id, 0) + (1.0 / (k_rrf + rank + 1))

        # --- STEP 3: FINAL FUSION ---
        # Sort by the combined RRF score
        final_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        
        return final_results[:]

    except Exception as e:
        print(f"Hybrid Search Error: {e}")
        conn.rollback()
        return []


In [341]:
def precision_search_weighted(query, query_embeddings, ids, top_k=3):
    inputs = processor(text=[query], return_tensors="pt")
    input_ids = inputs["input_ids"][0]
    query_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)

    raw_weights = get_query_weights(query_tokens)
    query_weights = np.power(raw_weights, 1)
    num_embeddings = query_embeddings.shape[0]
    num_tokens = len(query_weights)
    final_weights = np.ones(num_embeddings, dtype=np.float32)
    final_weights[:num_tokens] = query_weights
    
    try:
        all_page_data = {} 
        with conn.cursor() as cur:
            # We still only fetch patches for the candidate IDs
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        results = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32)
            
            # MaxSim calculation (The core of ColBERT/ColQwen)
            sim_matrix = query_embeddings @ page_matrix.T 
            token_scores = np.max(sim_matrix, axis=1) 
            
            # Apply query weights and calculate final scalar score for the page
            weighted_score = np.sum(token_scores * final_weights)
            
            results.append((p_id, float(weighted_score)))

        # Simply sort by score in descending order (No diversity penalty)
        results.sort(key=lambda x: x[1], reverse=True)
        
        return results[:top_k]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []


In [342]:
def precision_search_diverse_weighted(query, query_embeddings, ids, top_k=20, sat=1):
    inputs = processor(text=[query], return_tensors="pt")
    input_ids = inputs["input_ids"][0]
    query_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)

    raw_weights = get_query_weights(query_tokens)
    query_weights = np.power(raw_weights, 1)
    num_embeddings = query_embeddings.shape[0]
    num_tokens = len(query_weights)
    final_weights = np.ones(num_embeddings, dtype=np.float32)
    final_weights[:num_tokens] = query_weights
    
    try:
        all_page_data = {} # extraction of all patch embeddings per page 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32) # np matrix for page
            sim_matrix = query_embeddings @ page_matrix.T # matrix multiplication 
            token_scores = np.max(sim_matrix, axis=1) # sum best matching patches for page
            
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores * final_weights
            })

        # diversity selection (marginal gain logic)
        selected_pages = []
        best_scores_per_token = np.zeros(query_embeddings.shape[0]) # running coverage of query tokens

        for _ in range(min(top_k, len(scored_pages))):
            best_marginal_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                if page in selected_pages:
                    continue
                
                gains = np.maximum(0, page["token_scores"] - (best_scores_per_token)) # marginal gain
                marginal_gain = np.sum(gains)
                
                if marginal_gain > best_marginal_gain:
                    best_marginal_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                best_scores_per_token = np.maximum(best_scores_per_token, best_candidate["token_scores"]) # update running coverage
        
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []

In [343]:
def precision_search_diverse(query_embeddings, ids, top_k=10):
    
    try:
        all_page_data = {} # extraction of all patch embeddings per page 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32) # np matrix for page
            sim_matrix = query_embeddings @ page_matrix.T # matrix multiplication 
            token_scores = np.max(sim_matrix, axis=1) # sum best matching patches for page
            
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores
            })

        # diversity selection (marginal gain logic)
        selected_pages = []
        best_scores_per_token = np.zeros(query_embeddings.shape[0]) # running coverage of query tokens

        for _ in range(min(top_k, len(scored_pages))):
            best_marginal_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                if page in selected_pages:
                    continue
                
                gains = np.maximum(0, page["token_scores"] - best_scores_per_token) # marginal gain
                marginal_gain = np.sum(gains)
                
                if marginal_gain > best_marginal_gain:
                    best_marginal_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                best_scores_per_token = np.maximum(best_scores_per_token, best_candidate["token_scores"]) # update running coverage
        
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages], scored_pages

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []


In [344]:
import numpy as np

def precision_search_sigmoid_saturation(query, query_embeddings, ids, top_k=3):
    inputs = processor(text=[query], return_tensors="pt")
    input_ids = inputs["input_ids"][0]
    query_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)

    # 1. Weight Amplification: Cube the weights to aggressively 
    # separate high-IDF signal from low-IDF noise.
    raw_weights = get_query_weights(query_tokens)
    query_weights = np.power(raw_weights, 2) 
    
    num_embeddings = query_embeddings.shape[0]
    num_tokens = len(query_weights)
    final_weights = np.ones(num_embeddings, dtype=np.float32)
    final_weights[:num_tokens] = query_weights
    
    try:
        all_page_data = {} 
        with conn.cursor() as cur:
            # Note: Ensure "ids" is a list/tuple of integers or strings as per your DB
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32)
            # MaxSim calculation
            sim_matrix = query_embeddings @ page_matrix.T
            token_scores = np.max(sim_matrix, axis=1) 
            
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores * final_weights
            })

        # 2. Diversity Selection via Sigmoid Saturation
        selected_pages = []
        # Track 'saturation' per token (0.0 = not seen, 1.0 = fully covered)
        token_coverage = np.zeros(query_embeddings.shape[0]) 

        for _ in range(min(top_k, len(scored_pages))):
            best_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                # Proper duplicate check using ID
                if any(p["id"] == page["id"] for p in selected_pages):
                    continue
                
                # SIGMOID LOGIC: As coverage for a token approaches 1.0, 
                # its 'remaining potential' drops to 0.
                # This prevents Page 15's 'tender' match from killing Page 21's 'tribunal' match.
                remaining_potential = 1.0 / (1.0 + np.exp(5 * (token_coverage - 0.5)))
                
                # Gain is the scores this page provides multiplied by what's still 'needed'
                current_gains = page["token_scores"] * remaining_potential
                marginal_gain = np.sum(current_gains)
                
                if marginal_gain > best_gain:
                    best_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                # Update coverage map
                # We normalize by 1.5 (est max score) to keep coverage in 0-1 range
                norm_scores = best_candidate["token_scores"] / 1.5
                token_coverage = np.maximum(token_coverage, norm_scores)
        
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages]

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []


In [13]:
def retrieve_content(results):
    output = []
    with conn.cursor() as cur:
        for id, score in results:
            cur.execute('''
                SELECT page_number, markdown
                FROM pages
                WHERE id = %s
            ''', (id,))

            row = cur.fetchone()
            output.append({
                "page_number": row[0],
                "markdown": row[1]
            })
    return output

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load model and tokenizer directly
model_name = 'BAAI/bge-reranker-v2-m3'
tokenizer = AutoTokenizer.from_pretrained(model_name)
reranker = AutoModelForSequenceClassification.from_pretrained(model_name)

reranker.to(device)
reranker.eval()


def rerank(query, content):
    if not content:
        return []

    # 2. Prepare text pairs
    pairs = [[query, item["markdown"]] for item in content]
    
    with torch.no_grad():
        # 3. Tokenize manually - this avoids the 'prepare_for_model' glitch
        inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=8192)
        
        # Move to GPU if available
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # 4. Get logits (scores)
        scores = reranker(**inputs).logits.view(-1).float()
        
        # Move back to CPU and convert to list
        scores = scores.cpu().tolist()

    # 5. Attach scores and sort
    for i, score in enumerate(scores):
        content[i]["score"] = score

    content.sort(key=lambda x: x["score"], reverse=True)
    return content[:3]


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9921.10it/s]


In [15]:
def get_page_base64(pdf_path, page_number):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)

    zoom = 1.0 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("jpg")
    return base64.b64encode(img_bytes).decode('utf-8')


In [101]:
url = "https://api.jina.ai/v1/rerank"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

In [156]:
def jina_reranker(query, content):
    candidates = []
    for dic in content:
        page_num = dic["page_number"]
        candidates.append({
            "page": page_num,
            "img": get_page_base64(pdf_path, page_num),
        })
    data = {
        "model": "jina-reranker-m0",
        "query": query,
        "documents": [{"image": c["img"]} for c in candidates]
    }

    response = requests.post(url, headers=headers, json=data)
    results = response.json()
    final = []
    for item in results.get("results", []):
        original_index = item["index"]
        page_data = candidates[original_index]
        page_number = page_data["page"]
        final.append({
            "page_number": page_number,
            "score": item["relevance_score"]
        })

    
    return final

In [16]:
def generate_context(content, pdf_path):
    context = []
    for dic in content:
        context.append({
            "type": "text",
            "text": f"--- Page {dic['page_number']} ---\n{dic['markdown']}"
        })

        b64_img = get_page_base64(pdf_path, dic["page_number"])

        context.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpg;base64,{b64_img}"}
        })
    return HumanMessage(content=context)

In [17]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes the markdown and image of pages of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. MARKDOWN: Use the markdown text for precise data extractions, quotes and tables.
    3. IMAGES: Use the images to verify layout, check for visual elements, and clarify ambiguous parts of the text.
    4. DISCREPANCY: Treat the page image and markdown as equal independent sources of information. ANY CLAIM OR INFORMATION MUST BE SUPPORTED BY BOTH SOURCES. Flag any discrepancies. 
    5. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    6. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    7. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    8. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

TESTING

In [174]:
query = 'What is Singapore doing to prepare students for an AI-driven future?'


In [175]:
coarse_vector, embeddings = embed_query(query, model, processor)
dic = coarse_search(coarse_vector)
ids = [key for key in dic.keys()]
results = precision_search(embeddings, ids)
content = retrieve_content(results)
results = rerank(query, content)
results

[{'page_number': 60,
  'markdown': "<!-- image -->\n\n## ii.  Equipping workers with skills to succeed\n\nEmbracing digital technologies is crucial for staying relevant and competitive. Workers must be prepared to continually acquire new  digital  skills  that  could  open  new opportunities for growth and advancement.\n\nWe recognise, however, that rapid changes in the digital landscape can be overwhelming and daunting for workers. We are committed to helping workers develop the confidence and skills to navigate digital transformation so that everyone moves forward together in our digital future.\n\nOur workers are in fact in a strong position to benefit from further digital developments. Today, Singapore's workforce is highly skilled and digitally confident. The number of tech professionals has grown from 166,000 in 2018 to around 200,000 in 2024, driven by demand from both Information &amp; Communications (I&amp;C) and non-I&amp;C sectors.\n\nWe will continue to provide comprehensiv

In [157]:
result = jina_reranker(query, content)

In [158]:
result

[{'page_number': 16, 'score': 0.98602277},
 {'page_number': 19, 'score': 0.9831887},
 {'page_number': 21, 'score': 0.9831887},
 {'page_number': 20, 'score': 0.98212385},
 {'page_number': 18, 'score': 0.98212385},
 {'page_number': 13, 'score': 0.98040132},
 {'page_number': 6, 'score': 0.97645465},
 {'page_number': 7, 'score': 0.97497411},
 {'page_number': 3, 'score': 0.97340301},
 {'page_number': 8, 'score': 0.96996801}]

In [122]:
content

[{'page_number': 12,
  'markdown': "## Notification of purpose\n\nIt is important for organisations to notify individuals of the purposes of and obtain their consent for collecting, using and disclosing their personal data, unless any exception applies. To improve user-friendliness for consumers, notifications provided through ICT systems should be easy to understand, be provided dynamically and adopt a layered approach.\n\n| Notification of Purpose                                                                                                                                                                                                                                                     | Data Lifecycle   | Data Lifecycle   | Data Lifecycle   | Data Lifecycle   | Data Lifecycle   | Data Lifecycle   |\n|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [62]:
input

[334, 346, 347, 331, 332]

In [168]:
df = pd.read_csv('tests/test_set - Smart Nation.csv')

In [169]:
questions = df["Question"]

In [176]:
data = {"Question": [], "Response": [], "Page Number": []}
for query in questions:
    coarse_vector, embeddings = embed_query(query, model, processor)
    dic = coarse_search(coarse_vector)
    ids = [key for key in dic.keys()]
    results = precision_search(embeddings, ids)
    content = retrieve_content(results)
    reranked = rerank(query, content)
    context = generate_context(reranked, pdf_path)
    message = generate_message(query, context)
    response = llm.invoke(message).content
    data["Question"].append(query)
    data["Response"].append(response)
    data["Page Number"].append([c["page_number"] for c in reranked])

    
    

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree



In [177]:
df = pd.DataFrame(data)

In [178]:
df

,Question,Response,Page Number
0,What is Smart Nation 2.0 trying to achieve?,Smart Nation 2.0 aims to achieve three key goa...,"[28, 4, 91]"
1,What is Singapore's plan for internet speeds i...,Singapore plans to upgrade its Nationwide Broa...,"[15, 51, 6]"
2,How has Singapore's approach to cybersecurity ...,Singapore's approach to cybersecurity has evol...,"[33, 89, 40]"
3,How is Singapore helping seniors who live alon...,Singapore is implementing several initiatives ...,"[25, 72, 71]"
4,How significant is the digital sector to Singa...,The digital sector is highly significant to Si...,"[49, 50, 6]"
5,How has Singapore been helping seniors and haw...,Singapore has implemented several initiatives ...,"[71, 72, 70]"
6,What help is available for people who have bee...,### Help Available for Victims of Cyberbullyin...,"[42, 41, 46]"
7,What is Singapore's overall approach to keepin...,Singapore's overall approach to keeping the in...,"[31, 92, 40]"
8,What is Singapore doing to fight scams?,Singapore is implementing several measures to ...,"[37, 89, 38]"
9,How is AI being used in Singapore's healthcare...,AI is being utilized in Singapore's healthcare...,"[21, 55, 26]"


In [179]:
df.to_csv('results.csv', index=False)